# Lesson 24: Reinforcement Learning & Q-Learning (Peter and the Wolf)

Self-contained `Board` environment (`rlboard.py` embedded inline), `random_policy` benchmark, Bellman Q-Table training over 5,000 epochs, and policy evaluation.

In [ ]:
import enum
import random
import numpy as np
import matplotlib.pyplot as plt

class Board:
    class Cell(enum.Enum):
        empty = 0; water = 1; wolf = 2; tree = 3; apple = 4
    def __init__(self, width, height):
        self.width = width; self.height = height
        self.matrix = np.zeros((width, height), dtype=int)
        self.human = (0, 0)
    def randomize(self, seed=13):
        rng = random.Random(seed)
        self.matrix.fill(0)
        for _ in range(5): self.matrix[rng.randrange(self.width), rng.randrange(self.height)] = Board.Cell.water.value
        for _ in range(3): self.matrix[rng.randrange(self.width), rng.randrange(self.height)] = Board.Cell.wolf.value
        for _ in range(3): self.matrix[rng.randrange(self.width), rng.randrange(self.height)] = Board.Cell.apple.value
    def random_start(self):
        while True:
            x, y = random.randrange(self.width), random.randrange(self.height)
            if self.matrix[x, y] == 0: self.human = (x, y); break
    def is_valid(self, pos): return 0 <= pos[0] < self.width and 0 <= pos[1] < self.height
    def at(self, pos=None):
        p = pos or self.human
        return Board.Cell(self.matrix[p[0], p[1]])
    def move(self, dpos):
        self.human = (self.human[0] + dpos[0], self.human[1] + dpos[1])

actions = {'U': (0, -1), 'D': (0, 1), 'L': (-1, 0), 'R': (1, 0)}
action_idx = {a: i for i, a in enumerate(actions)}
m = Board(8, 8); m.randomize(seed=13)
Q = np.ones((8, 8, 4), dtype=float) * 0.25

def probs(v, eps=1e-4):
    v = v - v.min() + eps
    return v / v.sum()

for epoch in range(2000):
    m.random_start()
    cum_r = 0.0
    for n in range(100):
        x, y = m.human
        a = random.choices(list(actions), weights=probs(Q[x, y]))[0]
        m.move(actions[a])
        if not m.is_valid(m.human) or m.at() in (Board.Cell.water, Board.Cell.wolf):
            r = -10.0; done = True
        elif m.at() == Board.Cell.apple:
            r = 10.0; done = True
        else:
            r = -0.1; done = False
        ai = action_idx[a]
        nxt = 0.0 if done else Q[m.human[0], m.human[1]].max()
        Q[x, y, ai] = 0.7 * Q[x, y, ai] + 0.3 * (r + 0.5 * nxt)
        if done: break
print('Q-Table trained! Sample start Q-values:', np.round(Q[0, 0], 2))

## Assignment: A More Realistic World

Add a `max_steps=100` cap and visited-state tracking to `walk()` so a strict `np.argmax` policy never loops infinitely.